# Phase 1: Data Cleaning & Star Schema Pipeline
**Netflix Originals Analytics Portfolio Project**

This notebook documents the ingestion, transformation, feature engineering, and relational modeling of `NetflixOriginals.csv`.

### Business Context & Objectives:
- Client: Content Strategy Team at a streaming platform.
- Objective: Transform uncurated raw CSV into a clean star schema with robust date dimensions, normalized bridge tables for compound genres, and analytical categorical buckets.
- Target outputs: SQLite database (`netflix.db`) and CSV star schema in `data/processed/powerbi/`.

In [1]:
import os
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

# Relative paths
RAW_PATH = os.path.join("..", "NetflixOriginals.csv")
GENRE_MAP_PATH = os.path.join("..", "data", "external", "genre_group_map.csv")
PROCESSED_DIR = os.path.join("..", "data", "processed", "powerbi")
DB_PATH = os.path.join("..", "netflix.db")

print(f"Raw dataset path: {RAW_PATH}")

Raw dataset path: ..\NetflixOriginals.csv


## 1. Data Ingestion & Initial Audit
Read raw CSV with `encoding='latin-1'` as required by special characters in titles.

In [2]:
df_raw = pd.read_csv(RAW_PATH, encoding='latin-1')
print(f"Dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Missing values per column:\n{df_raw.isnull().sum()}")
print(f"Unique titles: {df_raw['Title'].nunique()}")
assert df_raw.shape[0] == 584, "Expected exactly 584 rows"
assert df_raw['Title'].nunique() == 584, "All titles must be unique"
df_raw.head(3)

Dataset shape: 584 rows, 6 columns
Missing values per column:
Title         0
Genre         0
Premiere      0
Runtime       0
IMDB Score    0
Language      0
dtype: int64
Unique titles: 584


,Title,Genre,Premiere,Runtime,IMDB Score,Language
0,Enter the Anime,Documentary,"August 5, 2019",58,2.5,English/Japanese
1,Dark Forces,Thriller,"August 21, 2020",81,2.6,Spanish
2,The App,Science fiction/Drama,"December 26, 2019",79,2.6,Italian


## 2. Date Parsing & Temporal Dimensions
- Handles formatting anomalies (5 rows have periods instead of commas, e.g. `October 16. 2019`).
- Generates Date Key (`YYYYMMDD`), Year, Quarter, Month Name, Day of Week, and flags.
- `is_full_year`: Flagged `1` for 2016-2020 (mature reporting period), `0` for sparse 2014-2015 and partial 2021.

In [3]:
# Clean periods and standardize
clean_dates = df_raw['Premiere'].str.replace('.', ',', regex=False).str.strip()
parsed_dates = pd.to_datetime(clean_dates, format='mixed')

df_clean = df_raw.copy()
df_clean['premiere_date'] = parsed_dates
df_clean['date_key'] = df_clean['premiere_date'].dt.strftime('%Y%m%d').astype(int)
df_clean['year'] = df_clean['premiere_date'].dt.year
df_clean['quarter'] = 'Q' + df_clean['premiere_date'].dt.quarter.astype(str)
df_clean['month_num'] = df_clean['premiere_date'].dt.month
df_clean['month_name'] = df_clean['premiere_date'].dt.month_name()
df_clean['day_of_month'] = df_clean['premiere_date'].dt.day
df_clean['day_of_week'] = df_clean['premiere_date'].dt.dayofweek + 1  # 1 = Monday
df_clean['weekday_name'] = df_clean['premiere_date'].dt.day_name()
df_clean['is_weekend'] = df_clean['day_of_week'].isin([6, 7]).astype(int)
df_clean['is_full_year'] = df_clean['year'].between(2016, 2020).astype(int)

print("Releases by year with full-year flag:")
print(df_clean.groupby(['year', 'is_full_year']).size().reset_index(name='title_count'))

Releases by year with full-year flag:
   year  is_full_year  title_count
0  2014             0            1
1  2015             0            9
2  2016             1           30
3  2017             1           66
4  2018             1           99
5  2019             1          125
6  2020             1          183
7  2021             0           71


## 3. Genre Normalization, Bridge Table & Genre Grouping
- 51 rows contain compound genres separated by `/`.
- We construct `bridge_title_genre` to preserve many-to-many relationships without duplicating title metrics.
- Primary genre rule: The first listed genre is designated as `is_primary = 1`.
- Map all raw genres into 8 high-level analytical genre groups via `data/external/genre_group_map.csv`.

In [4]:
genre_map_df = pd.read_csv(GENRE_MAP_PATH)
genre_group_lookup = dict(zip(genre_map_df['raw_genre'], genre_map_df['genre_group']))

# Assign Title ID
df_clean['title_id'] = range(1, len(df_clean) + 1)

# Parse genres
bridge_rows = []
atomic_genres = set()

for _, row in df_clean.iterrows():
    t_id = row['title_id']
    raw_g = row['Genre'].strip()
    parts = [p.strip() for p in raw_g.split('/') if p.strip()]
    for idx, g in enumerate(parts):
        atomic_genres.add(g)
        bridge_rows.append({
            'title_id': t_id,
            'genre_name': g,
            'is_primary': 1 if idx == 0 else 0
        })

df_bridge = pd.DataFrame(bridge_rows)

# Create dim_genre
sorted_genres = sorted(list(atomic_genres))
genre_to_id = {g: i + 1 for i, g in enumerate(sorted_genres)}
dim_genre = pd.DataFrame([
    {
        'genre_id': genre_to_id[g],
        'genre_name': g,
        'genre_group': genre_group_lookup.get(g, 'Other')
    }
    for g in sorted_genres
])

df_bridge['genre_id'] = df_bridge['genre_name'].map(genre_to_id)
df_bridge = df_bridge[['title_id', 'genre_id', 'is_primary']]

# Primary genre info for fact table
primary_genres = df_clean['Genre'].apply(lambda x: [p.strip() for p in x.split('/')][0])
df_clean['primary_genre_name'] = primary_genres
df_clean['primary_genre_id'] = df_clean['primary_genre_name'].map(genre_to_id)
df_clean['genre_group'] = df_clean['Genre'].map(genre_group_lookup).fillna(df_clean['primary_genre_name'].map(lambda g: genre_group_lookup.get(g, 'Other')))

print(f"Unique atomic genres in dim_genre: {len(dim_genre)}")
print(f"Total bridge rows: {len(df_bridge)}")
print("\nDistribution across 8 Genre Groups:")
print(df_clean['genre_group'].value_counts())

Unique atomic genres in dim_genre: 91
Total bridge rows: 643

Distribution across 8 Genre Groups:
genre_group
Documentary                 163
Drama                        98
Comedy                       90
Thriller/Crime/Horror        85
Romance                      57
Sci-Fi/Fantasy/Animation     44
Music/Concert/Special        28
Action/Adventure/Other       19
Name: count, dtype: int64


## 4. Language Normalization & Multilingual Flags
- 23 rows contain compound languages separated by `/`.
- We extract `primary_language` (first listed) and set `is_multilingual = 1` for compound entries.

In [5]:
lang_parts = df_clean['Language'].apply(lambda x: [p.strip() for p in x.split('/') if p.strip()])
df_clean['primary_language'] = lang_parts.apply(lambda parts: parts[0])
df_clean['is_multilingual'] = lang_parts.apply(lambda parts: 1 if len(parts) > 1 else 0)

unique_langs = sorted(df_clean['primary_language'].unique())
lang_to_id = {l: i + 1 for i, l in enumerate(unique_langs)}
dim_language = pd.DataFrame([
    {'language_id': lang_to_id[l], 'language_name': l}
    for l in unique_langs
])
df_clean['primary_language_id'] = df_clean['primary_language'].map(lang_to_id)

print(f"Primary languages count: {len(dim_language)}")
print(f"Multilingual titles: {df_clean['is_multilingual'].sum()}")
print("\nTop languages:")
print(df_clean['primary_language'].value_counts().head(6))

Primary languages count: 24
Multilingual titles: 23

Top languages:
primary_language
English       419
Spanish        34
Hindi          33
French         20
Italian        14
Portuguese     12
Name: count, dtype: int64


## 5. Analytical Bucketing & Feature Engineering
We define two standard business categorical dimensions:
1. **Runtime Buckets**:
   - `Short/Special (<40m)`: Format variance (shorts, specials, aftershows).
   - `Standard (40-120m)`: Standard feature release.
   - `Long (>120m)`: Extended/epic release.
2. **Score Buckets**:
   - `Low (<5.5)`: Sub-par quality floor.
   - `Mid (5.5-6.9)`: Solid average quality.
   - `High (>=7.0)`: Acclaimed / marquee titles.

In [6]:
def assign_runtime_bucket(rt):
    if rt < 40:
        return 'Short/Special (<40m)'
    elif rt <= 120:
        return 'Standard (40-120m)'
    return 'Long (>120m)'

def assign_score_bucket(sc):
    if sc < 5.5:
        return 'Low (<5.5)'
    elif sc < 7.0:
        return 'Mid (5.5-6.9)'
    return 'High (>=7.0)'

df_clean['runtime_bucket'] = df_clean['Runtime'].apply(assign_runtime_bucket)
df_clean['score_bucket'] = df_clean['IMDB Score'].apply(assign_score_bucket)

print("Runtime bucket distribution:")
print(df_clean['runtime_bucket'].value_counts())
print("\nScore bucket distribution:")
print(df_clean['score_bucket'].value_counts())

Runtime bucket distribution:
runtime_bucket
Standard (40-120m)      474
Long (>120m)             68
Short/Special (<40m)     42
Name: count, dtype: int64

Score bucket distribution:
score_bucket
Mid (5.5-6.9)    325
High (>=7.0)     152
Low (<5.5)       107
Name: count, dtype: int64


## 6. Star Schema Construction & Validation
Construct and validate `fact_titles`, `dim_date`, `dim_genre`, `bridge_title_genre`, and `dim_language`.

In [7]:
# Fact table
fact_titles = df_clean[[
    'title_id', 'Title', 'date_key', 'primary_genre_id', 'primary_genre_name',
    'genre_group', 'primary_language_id', 'primary_language', 'is_multilingual',
    'Runtime', 'runtime_bucket', 'IMDB Score', 'score_bucket'
]].rename(columns={'Title': 'title', 'Runtime': 'runtime', 'IMDB Score': 'imdb_score'})

# Dim date
dim_date = df_clean[[
    'date_key', 'premiere_date', 'year', 'quarter', 'month_num', 'month_name',
    'day_of_month', 'day_of_week', 'weekday_name', 'is_weekend', 'is_full_year'
]].drop_duplicates(subset=['date_key']).rename(columns={'premiere_date': 'full_date'}).sort_values('date_key')
dim_date['full_date'] = dim_date['full_date'].dt.strftime('%Y-%m-%d')

# Data validation assertions
assert len(fact_titles) == 584, "Fact titles count must be exactly 584"
assert fact_titles['title_id'].nunique() == 584, "title_id must be unique PK"
assert df_bridge['title_id'].isin(fact_titles['title_id']).all(), "No orphan title IDs in bridge"
assert df_bridge['genre_id'].isin(dim_genre['genre_id']).all(), "No orphan genre IDs in bridge"
assert fact_titles['primary_language_id'].isin(dim_language['language_id']).all(), "No orphan languages"
assert fact_titles['imdb_score'].between(0, 10).all(), "Scores must be within [0, 10]"

print("ALL INTEGRITY & REFERENTIAL CONSTRAINTS PASSED SUCCESSFULLY!")
print(f"fact_titles rows: {len(fact_titles)}")
print(f"dim_date rows: {len(dim_date)}")
print(f"dim_genre rows: {len(dim_genre)}")
print(f"bridge_title_genre rows: {len(df_bridge)}")
print(f"dim_language rows: {len(dim_language)}")

ALL INTEGRITY & REFERENTIAL CONSTRAINTS PASSED SUCCESSFULLY!
fact_titles rows: 584
dim_date rows: 387
dim_genre rows: 91
bridge_title_genre rows: 643
dim_language rows: 24
